## Import Libraries

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
import config

## Functions

In [5]:
def load_yahoo_data(ticker, input_dir):
    """
    Load data (date, prices, dividend and volume) from cleaned Yahoo Finance CSV file.
    """

    input_file = input_dir / f"{ticker}.csv"

    if not input_file.exists():
        raise FileNotFoundError(f"Yahoo Finance file not found for {ticker}: {input_file}")

    data = pd.read_csv(input_file)
    data["Date"] = pd.to_datetime(data["Date"])
    data = data.sort_values("Date").reset_index(drop=True)
    data["Ticker"] = ticker

    # Keep Monday-Friday observations only (remove weekend days for crypto assets)
    data = data[data["Date"].dt.dayofweek < 5].copy()

    return data

def construct_return_momentum_features(data):
    """
    Construct daily returns and momentum features for one asset.
    """

    data = data.copy()
    price = data["Adj Close"]

    # Daily return
    data["Return_1d"] = price.pct_change()

    # Momentum returns
    for window in config.MOMENTUM_WINDOWS:
        data[f"Momentum_{window}d"] = (price / price.shift(window) - 1)

    return data

def construct_trend_features(data):
    """
    Construct moving-average based trend features.
    """

    data = data.copy()
    price = data["Adj Close"]

    # Moving averages
    for window in config.TREND_WINDOWS:
        data[f"MA_{window}d"] = (price.rolling(window=window).mean())

        # Relative distance from moving average
        data[f"Price_to_MA_{window}d"] = (price / data[f"MA_{window}d"] - 1)

    # Moving-average crossover signals
    data["MA_20_50_Spread"] = (data["MA_20d"] / data["MA_50d"] - 1)
    data["MA_50_200_Spread"] = (data["MA_50d"] / data["MA_200d"] - 1)

    return data

def construct_mean_reversion_features(data):
    """
    Construct rolling mean-reversion and z-score features.
    """

    data = data.copy()
    price = data["Adj Close"]

    for window in config.MEAN_REVERSION_WINDOWS:

        rolling_mean = price.rolling(window=window).mean()
        rolling_std = price.rolling(window=window).std()

        # Relative distance from rolling mean
        data[f"Mean_Reversion_{window}d"] = (price / rolling_mean - 1)

        # Price z-score
        data[f"Price_ZScore_{window}d"] = ((price - rolling_mean) / rolling_std)

    return data

def construct_volatility_features(data):
    """
    Construct rolling annualized realized-volatility features.
    """

    data = data.copy()
    returns = data["Return_1d"]

    for window in config.VOLATILITY_WINDOWS:

        data[f"Volatility_{window}d"] = (returns.rolling(window=window).std()* np.sqrt(252))

    return data

def construct_drawdown_features(data):
    """
    Construct rolling drawdown features.
    """

    data = data.copy()
    price = data["Adj Close"]

    for window in config.DRAWDOWN_WINDOWS:

        rolling_high = price.rolling(window=window).max()
        data[f"Drawdown_{window}d"] = (price / rolling_high - 1)

    return data

def construct_volume_features(data):
    """
    Construct normalized volume features.
    """

    data = data.copy()

    volume = data["Volume"]

    for window in config.VOLUME_WINDOWS:

        volume_mean = volume.rolling(window=window).mean()
        volume_std = volume.rolling(window=window).std()

        # Current volume relative to average volume
        data[f"Volume_Ratio_{window}d"] = (volume / volume_mean)

        # Volume z-score
        data[f"Volume_ZScore_{window}d"] = ((volume - volume_mean) / volume_std)

    return data

def construct_target(data):
    """
    Construct the forward-return classification target.

    Target:
        1 = positive forward return
        0 = zero or negative forward return
    """

    data = data.copy()
    price = data["Adj Close"]

    # Forward return
    data[f"Forward_Return_{config.FORWARD_HORIZON}d"] = (price.shift(-config.FORWARD_HORIZON) / price - 1)

    # Classification target
    data[f"Target_{config.FORWARD_HORIZON}d"] = (data[f"Forward_Return_{config.FORWARD_HORIZON}d"] > 0).astype("Int64")

    return data

## Create Yahoo Features

In [6]:
# Define input data directories
YAHOO_INPUT_DIR = config.INPUT_DATA_DIR / "yahoo"
FRED_INPUT_DIR = config.INPUT_DATA_DIR / "fred"
FAMA_FRENCH_INPUT_DIR = config.INPUT_DATA_DIR / "fama_french"

# Research data output directory
FEATURES_DATA_DIR = config.INPUT_DATA_DIR.parent / "features_data"
FEATURES_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Output files
FEATURES_FILE = FEATURES_DATA_DIR / "asset_features.csv"
ELIGIBILITY_FILE = FEATURES_DATA_DIR / "asset_eligibility.csv"

# Load the master investable universe
universe = pd.read_csv(config.UNIVERSE_FILE)

# Get Yahoo Finance instruments
yahoo_universe = universe[universe["Source"].str.strip().str.lower() == "yahoo finance"].copy()
tickers = yahoo_universe["Ticker"].tolist()

# Construct features for all Yahoo assets
ticker_feature_data = []

for ticker in tickers:

    print(f"Constructing features for {ticker}...")

    data = load_yahoo_data(ticker=ticker, input_dir=YAHOO_INPUT_DIR)
    data = construct_return_momentum_features(data)
    data = construct_trend_features(data)
    data = construct_mean_reversion_features(data)
    data = construct_volatility_features(data)
    data = construct_drawdown_features(data)
    data = construct_volume_features(data)
    data = construct_target(data)

    ticker_feature_data.append(data)

# Combine assets and apply start date
yahoo_features = pd.concat(ticker_feature_data, ignore_index=True)
yahoo_features = yahoo_features.sort_values(["Date", "Ticker"]).reset_index(drop=True)
yahoo_features = yahoo_features[yahoo_features["Date"] >= config.START_DATE].copy()

# Keep features and target only
yahoo_features = (yahoo_features.drop(
    columns = ["Adj Close", "Close", "Capital Gains", "Dividends", "High", "Low", "Open", "Stock Splits", "Volume"]
).copy())

# Remove rows with missing values
yahoo_features = yahoo_features.dropna(subset = ["Momentum_252d", f"Forward_Return_{config.FORWARD_HORIZON}d"]).copy()

# Reset index after filtering
yahoo_features = yahoo_features.reset_index(drop=True)

# Save the file
yahoo_features.to_csv(FEATURES_FILE, index=False)
print(f"Saved asset features to: {FEATURES_FILE}")

Constructing features for SPY...
Constructing features for VT...
Constructing features for QQQ...
Constructing features for VTV...
Constructing features for VUG...
Constructing features for IWM...
Constructing features for DVY...
Constructing features for VEA...
Constructing features for VGK...
Constructing features for FEZ...
Constructing features for EWJ...
Constructing features for FXI...
Constructing features for AFK...
Constructing features for VWO...
Constructing features for SHY...
Constructing features for IEF...
Constructing features for TLT...
Constructing features for BND...
Constructing features for TIP...
Constructing features for VCSH...
Constructing features for LQD...
Constructing features for HYG...
Constructing features for EMB...
Constructing features for BWX...
Constructing features for VNQ...
Constructing features for RWX...
Constructing features for IGF...
Constructing features for GLD...
Constructing features for SLV...
Constructing features for USO...
Constructi